# StockStory India — Fine-Tune 0.5B SLM on Indian Market Data
**Dataset: 87 examples across 6 categories**

Run this on **free T4 GPU** in Colab. Runtime → Change runtime type → T4 GPU

---
The dataset is fetched automatically from GitHub. No manual upload needed.

In [ ]:
# ── Install Unsloth and dependencies ──
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets requests
print("\u2713 Dependencies installed")

In [ ]:
import os, json, requests

# Download dataset from GitHub raw
url = "https://raw.githubusercontent.com/samvidh75/PREDICTION-ENGINE/main/master_indian_market_train.json"
resp = requests.get(url)
if resp.status_code == 200:
    with open('master_indian_market_train.json', 'w') as f:
        f.write(resp.text)
    data = json.loads(resp.text)
    print(f"\u2713 Dataset downloaded: {len(data)} examples")
    cats = {}
    for d in data:
        inst = d['instruction'].lower()
        if 'fundamental' in inst: c = 'fundamental'
        elif 'technical' in inst: c = 'technical'
        elif 'regulatory' in inst: c = 'regulatory'
        elif 'rotation' in inst: c = 'rotation'
        elif 'earnings' in inst: c = 'earnings'
        elif 'corporate action' in inst: c = 'corporate_action'
        else: c = 'other'
        cats[c] = cats.get(c, 0) + 1
    for k, v in cats.items():
        print(f"    {k}: {v}")
else:
    print(f"\u274c Download failed: HTTP {resp.status_code}. Check internet connection.")

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("\u2713 Base model loaded")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("\u2713 LoRA adapters attached")

In [ ]:
prompt_format = """<|im_start|>system
You are a dedicated Indian stock market analyst. Use the context to provide a concise Healthometer assessment with score and risk level.
<|im_start|>user
Task: {}
Context: {}
<|im_start|>assistant
{}<|im_end|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        texts.append(prompt_format.format(inst, inp, out))
    return {"text": texts}

dataset = load_dataset("json", data_files="master_indian_market_train.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\u2713 Dataset loaded: {len(dataset)} examples")

In [ ]:
# 180 steps for 87 examples — ~10 min on T4
steps = 180

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=steps,
        learning_rate=3e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir="outputs",
        save_steps=steps,
    ),
)
print(f"\u2713 Trainer initialized ({steps} steps)")

In [ ]:
trainer_stats = trainer.train()
print("\u2713 Training complete!")

In [ ]:
# Save merged 16-bit model for deployment
model.save_pretrained_merged("indian_stock_slm_master", tokenizer, save_method="merged_16bit")
print("\u2713 Merged model saved to 'indian_stock_slm_master'")

# Export to GGUF for web deployment
model.save_pretrained_gguf("indian_stock_slm_master_gguf", tokenizer, quantization_method="q4_k_m")
print("\u2713 GGUF model saved for web deployment")

!ls -lh indian_stock_slm_master_gguf/

## Test the Fine-Tuned Model
Run inference on 12 benchmark stocks to verify Healthometer accuracy.

In [ ]:
FastLanguageModel.for_inference(model)

benchmark = [
    ("RELIANCE",    "Evaluate corporate governance and fundamental sustainability for an Indian equity.",
     "Ticker: RELIANCE | Sector: Energy/Telco/Retail | P/E: 32.4 | Debt/Equity: 0.8 | Promoter Pledging: 0% | ROCE: 16.2% | Revenue Growth: 12% | Auditor Commentary: Standard audit opinion.", 60, 75),
    ("ADANIENT",    "Evaluate corporate governance and fundamental sustainability for an Indian equity.",
     "Ticker: ADANIENT | Sector: Diversified | P/E: 45.2 | Debt/Equity: 1.8 | Promoter Pledging: 52.4% | ROCE: 8.5% | Revenue Growth: 22% | Auditor Commentary: Auditor noted multi-layered shell entities.", 20, 40),
    ("INFY",        "Evaluate corporate governance and fundamental sustainability for an Indian equity.",
     "Ticker: INFY | Sector: IT | P/E: 22.1 | Debt/Equity: 0.1 | Promoter Pledging: 0% | ROCE: 32.4% | Revenue Growth: 5% | Auditor Commentary: No material discrepancies. Revenue recognition policy consistent.", 75, 90),
    ("TCS",         "Evaluate corporate governance and fundamental sustainability for an Indian equity.",
     "Ticker: TCS | Sector: IT | P/E: 26.3 | Debt/Equity: 0.05 | Promoter Pledging: 0% | ROCE: 48.2% | Revenue Growth: 8% | Auditor Commentary: Clean audit with robust internal financial controls.", 80, 90),
    ("TATAMOTORS",  "Analyze quarterly earnings performance and management commentary.",
     "Ticker: TATAMOTORS | Sector: Auto | Result: Beat | Revenue Growth: 18.5% | Margin Change: 250bps | Guidance: Raised | Management Commentary: JLR margins at 9-year high.", 80, 95),
    ("WIPRO",       "Analyze quarterly earnings performance and management commentary.",
     "Ticker: WIPRO | Sector: IT | Result: Miss | Revenue Growth: 2.1% | Margin Change: -80bps | Guidance: Cut | Management Commentary: Client discretionary spending remains weak.", 25, 45),
    ("RELIANCE",    "Analyze technical price structures and institutional order flow anomalies on the NSE.",
     "Ticker: RELIANCE | Sector: Energy | Last Price: 2490 | Volume Expansion: 2.6x | Delivery %: 72% | Order Flow: Institutional block deal detected.", 75, 90),
    ("ADANIPORTS",  "Analyze technical price structures and institutional order flow anomalies on the NSE.",
     "Ticker: ADANIPORTS | Sector: Infrastructure | Last Price: 1150 | Volume Expansion: 1.4x | Delivery %: 32% | Order Flow: Continued delivery selling. DIIs reducing stake.", 30, 50),
    ("ZOMATO",      "Analyze technical price structures and institutional order flow anomalies on the NSE.",
     "Ticker: ZOMATO | Sector: Internet | Last Price: 268 | Volume Expansion: 1.8x | Delivery %: 76% | Order Flow: Mutual funds increased allocation.", 65, 85),
    ("WIPRO",       "Analyze technical price structures and institutional order flow anomalies on the NSE.",
     "Ticker: WIPRO | Sector: IT | Last Price: 520 | Volume Expansion: 1.2x | Delivery %: 33% | Order Flow: Contributed delivery selling post weak Q4.", 30, 50),
    ("Midcap Derivatives", "Assess the structural impact of a SEBI regulatory circular on market mechanics.",
     "Sector: Midcap Derivatives | SEBI Action: SEBI increases minimum contract value to 15 Lakhs and restricts weekly expiries. | Impacted Assets: Volatility Index, Nifty Midcap Select", 20, 45),
    ("IPO Framework", "Assess the structural impact of a SEBI regulatory circular on market mechanics.",
     "Sector: IPO Framework | SEBI Action: SEBI relaxes anchor investor lock-in period from 90 to 30 days for mainboard IPOs. | Impacted Assets: Primary market pipeline, HNI investors", 65, 85),
]

import re, time
correct = 0
total_time = 0

print(f"{'='*70}")
print(f"  BENCHMARK EVALUATION — {len(benchmark)} stocks")
print(f"{'='*70}")

for ticker, inst, inp, lo, hi in benchmark:
    prompt = prompt_format.format(inst, inp, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3)
    elapsed = time.time() - start
    total_time += elapsed
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_part = response.split("assistant")[-1].strip() if "assistant" in response else response
    match = re.search(r'Healthometer[:\s]*(\d+)/100', assistant_part)
    score = int(match.group(1)) if match else None
    is_correct = lo <= score <= hi if score else False
    if is_correct: correct += 1
    status = "\u2705" if is_correct else "\u274c"
    print(f"  {status} {ticker:<16} HO={score or 'N/A':<4} (expected {lo}-{hi}) [{elapsed:.1f}s]")

print()
print(f"{'='*70}")
print(f"  Accuracy: {correct}/{len(benchmark)} ({correct/len(benchmark)*100:.0f}%)")
print(f"  Avg time: {total_time/len(benchmark):.2f}s per query")
print(f"{'='*70}")
print(f"\nTo evaluate against the Cloudflare rule-based engine:\n")
print(f"  curl -X POST https://stockstory-pass-api.peridot-cough.workers.dev/api/v1/evaluate \\")
print(f"    -H 'Content-Type: application/json' -d '{{\"instruction\": \"...\", \"input\": \"...\"}}'")

---
**Model ready!** Download from the file sidebar:
1. `indian_stock_slm_master/` — merged 16-bit model (~90MB)
2. `indian_stock_slm_master_gguf/` — GGUF q4_k_m (~90MB, for web deployment)

**Next:** Upload GGUF as static asset to Cloudflare Pages.